<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/47_feedback_learning_agent/feedback_learning_agent_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re

In [2]:
# store corrected answers
feedback_memory = {}

In [3]:
def classify_intent(query):
    query = query.lower()

    if any(word in query for word in ["%", "calculate", "sum", "add", "multiply"]):
        return "calculation"
    elif any(word in query for word in ["what is", "who is", "define"]):
        return "knowledge"
    else:
        return "explanation"

In [4]:
def calculator_tool(query):
    match = re.search(r'(\d+)%.*?(\d+)', query)
    if match:
        percent = float(match.group(1))
        number = float(match.group(2))
        return (percent / 100) * number
    return "Calculation failed"


def knowledge_tool(query):
    if "artificial intelligence" in query.lower():
        return "Artificial Intelligence is the simulation of human intelligence in machines."
    return "Knowledge not found"


def explanation_tool(query):
    if "machine learning" in query.lower():
        return "Machine Learning is a subset of AI that learns from data."
    return "Explanation not found"

In [5]:
def check_feedback(query):
    return feedback_memory.get(query, None)


def store_feedback(query, correct_answer):
    feedback_memory[query] = correct_answer

In [6]:
def feedback_agent(query):

    # check if corrected answer exists
    corrected = check_feedback(query)
    if corrected:
        return {
            "query": query,
            "source": "feedback_memory",
            "answer": corrected
        }

    intent = classify_intent(query)

    if intent == "calculation":
        answer = calculator_tool(query)
    elif intent == "knowledge":
        answer = knowledge_tool(query)
    else:
        answer = explanation_tool(query)

    return {
        "query": query,
        "source": "tool_generated",
        "answer": answer
    }

In [7]:
query = input("Enter your query: ")

result = feedback_agent(query)
print("\nAnswer:", result["answer"])

feedback = input("\nIs this correct? (yes/no): ")

if feedback.lower() == "no":
    correct_answer = input("Please provide correct answer: ")
    store_feedback(query, correct_answer)
    print("\nFeedback stored! Try again.")

    # show improved result
    print("\nUpdated Answer:", feedback_agent(query)["answer"])

Enter your query: Explain deep learning

Answer: Explanation not found

Is this correct? (yes/no): no
Please provide correct answer: Deep Learning is a subset of machine learning that uses neural networks with multiple layers.

Feedback stored! Try again.

Updated Answer: Deep Learning is a subset of machine learning that uses neural networks with multiple layers.
